# FSOT Biohub v49 — CPU-only submission

Competition runtime has **no GPU**. This notebook forces CPU and runs the FSOT scalar engine (`fsot` + `peaks`) by default.

**Inputs:** competition test data + optional `fsot-v49-cpu-bundle` dataset with this repo's `v49/` Python files.

Lean verification ref: https://github.com/dappalumbo91/FSOT-2.1-Lean

In [ ]:
import os
import shutil
import glob
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)

# CPU-only — competition has no GPU workers
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ.setdefault("CELLMOT_DEVICE", "cpu")
os.environ.setdefault("BIOHUB_ENGINE", "fsot")
os.environ.setdefault("BIOHUB_DETECTOR", "peaks")
os.environ.setdefault("CELLMOT_USE_ILP", "0")
os.environ.setdefault("FSOT_LINK_MODE", "fsot_gate")
os.environ.setdefault("FSOT_GATE_FRAC", "0.42")
os.environ.setdefault("FSOT_GAP_LINK", "1")
os.environ.setdefault("KAGGLE_SUBMISSION_FAST_VALIDATE", "0")

print("Kaggle input:", os.listdir("/kaggle/input"))

# Copy v49 bundle from attached dataset (if present)
_bundle_hits = glob.glob("/kaggle/input/**/kaggle_main_runner_cpu.py", recursive=True)
if _bundle_hits:
    src_dir = Path(_bundle_hits[0]).parent
    for name in [
        "kaggle_main_runner_cpu.py",
        "fsot_original_competition.py",
        "fsot_core.py",
        "fsot_cellular_bridge.py",
        "biohub_competitive.py",
        "biohub_unet_engine.py",
        "submission_io.py",
        "validate_kaggle_submission.py",
        "csv_to_geffs.py",
    ]:
        p = src_dir / name
        if p.exists():
            shutil.copy2(p, WORK / name)
    print(f"Copied v49 bundle from {src_dir}")
else:
    print("WARN: fsot-v49-cpu-bundle not found — expecting files already in /kaggle/working")

# Optional offline wheels (zarr/dask only — no torch for default fsot engine)
_offline = glob.glob("/kaggle/input/**/zarr-*.whl", recursive=True)
if _offline:
    wheel_dir = os.path.dirname(_offline[0])
    subprocess.run(
        f"pip install --no-index --no-deps --find-links {wheel_dir} zarr dask",
        shell=True, capture_output=True, text=True,
    )
    print(f"Offline wheels: {wheel_dir}")

In [ ]:
import runpy
import sys

sys.path.insert(0, "/kaggle/working")
runpy.run_path("/kaggle/working/kaggle_main_runner_cpu.py", run_name="__main__")

import pandas as pd
sub = pd.read_csv("/kaggle/working/submission.csv")
print(sub.groupby(["dataset", "row_type"]).size())
print(f"submission rows: {len(sub)}")